# Project 2

We compare two pretrained word embedding models:

- GloVe WikiGiga 50d: general domain embeddings
- GloVe Twitter 50d: Twitter domain embeddings

We also compare different tokenizer configurations, change punctuation and stopword removal settings.

Goal: investigate whether domain-specific pretrained embeddings provide better vocabulary coverage, semantic representations and emotion classification performance.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys

from pathlib import Path
from datasets import load_dataset

sys.path.insert(0, str(Path.cwd().parent))

from part2.embeddings import (
    load_embeddings,
    create_tokenizer,
    vocab_coverage,
    cos_similarity,
    topk_ids,
    train_embed_pool
)

In [4]:
wiki = load_embeddings("wikigiga")
twitter = load_embeddings("twitter")

models = {
    "WikiGiga": wiki,
    "Twitter": twitter
}


In [5]:
dataset = load_dataset("cardiffnlp/tweet_eval", "emotion")

train_texts = dataset["train"]["text"]
val_texts = dataset["validation"]["text"]

labels = dataset["train"]["label"]

label_names = dataset["train"].features["label"].names

print(label_names)
print(dataset)

['anger', 'joy', 'optimism', 'sadness']
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 3257
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1421
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 374
    })
})


# Question 1. Tokenizer compatibility
Task: Find a word embedding model that is appropriate for your dataset. Think about the language, the kind of text you are working with. Find evidence that your choice is a good one: 
- Can you tokenize your texts in a way coherent with the definition of a token in the embedding model?
- Does the model's vocabulary give enough coverage to the words in your texts? etc.

To answer this we compare the vocabulary coverage of GloVe WikiGiga and
GloVe Twitter on the TweetEval training dataset.

We measure:
- Token coverage: percentage of all token occurrences represented.
- Unique-token coverage: percentage of distinct tokens represented.
- Most frequent out-of-vocabulary tokens.

We hypothesize that Twitter-trained embeddings might be better
suited to informal social-media language.

In [82]:
coverage_results = []
unknown_words = {}

for name in ["wikigiga", "twitter"]:
    model = load_embeddings(name)
    tokenizer = create_tokenizer(model)

    result = vocab_coverage(train_texts, tokenizer, model)

    coverage_results.append({
        "model": name,
        "token_coverage": result["token_coverage"],
        "unique_coverage": result["unique_coverage"]
    })

    unknown_words[name] = result["top_unknown"]

    del tokenizer
    del model

coverage_df = pd.DataFrame(coverage_results)
coverage_df

,model,token_coverage,unique_coverage
0,wikigiga,0.966265,0.821063
1,twitter,0.957949,0.816455


**1.1 Interpreting vocabulary coverage**

Token coverage is the proportion of all token occurrences for which a pretrained embedding exists. Unique-token coverage is the proportion of different token types that have an embedding. For example, if a common word appears 100 times, it contributes 100 times to token coverage but only once to unique-token coverage.

With our default BasicTokenizer settings (lowercasing, preserving punctuation, preserving stopwords), WikiGiga covers 96.63% of token occurrences and 82.11% of unique tokens, while Twitter covers 95.79% and 81.65%. Both cover most occurrences, but WikiGiga is slightly better on these measures. Coverage alone does not establish which model will classify emotion more accurately.


**1.2 Investigating out-of-vocabulary (OOV) tokens**

An OOV token is a token produced by our tokenizer that is not in the pretrained model's vocabulary. The coverage percentages tell us how many tokens are missing, but not which ones. Below are the out-of-vocabulary tokens displayed:

In [83]:
for model_name, oov_list in unknown_words.items():
    print(f"\nMost frequent OOV tokens: {model_name}")

    display(pd.DataFrame(
        oov_list,
        columns=["OOV token", "Frequency"]
    ))


Most frequent OOV tokens: wikigiga


,OOV token,Frequency
0,..,82
1,gbbo,26
2,\n,24
3,😂,22
4,bb18,15
5,lmao,14
6,dont,12
7,mhchat,11
8,🙄,10
9,\n\n,10



Most frequent OOV tokens: twitter


,OOV token,Frequency
0,...,196
1,..,82
2,2,61
3,....,43
4,3,29
5,live.ly,27
6,\n,24
7,1,23
8,😂,22
9,4,19


**Interpretation**

The most frequent OOV tokens for both models include punctuation sequences, emojis, newline characters, informal expressions, and uncommon words.

WikiGiga has several missing emojis and informal expressions, such as lmao, dont, and y'all. This is potentially problematic for emotion classification because emojis and informal expressions can carry important emotional information.

Twitter GloVe also fails to cover several emojis and punctuation sequences, despite being trained on Twitter data. Its most frequent OOV tokens additionally include numbers and live.ly.

Overall, both models struggle with some characteristics of informal social media language. Although WikiGiga has slightly higher overall vocabulary coverage, this does not necessarily mean it captures emotional information better. The types of missing tokens are also important for evaluating suitability for emotion classification.

**1.3 Does preprocessing affect coverage?**

We compare three tokenization choices using both embedding models:
1. Baseline: lowercase text; preserve punctuation and stopwords.
2. Remove punctuation: same as baseline except punctuation is removed.
3. Preserve case: retain capital letters instead of lowercasing.
Removing punctuation could increase measured coverage while simultaneously discarding cues such as !, ? or emoticons. We do not remove stopwords in our baseline because words such as not and never may affect emotion.

In [ ]:
# Compare preprocessing choices on the same training tweets.
preprocessing_settings = {
    "Baseline": dict(lowercase=True, remove_punct=False, remove_stopwords=False),
    "Remove punctuation": dict(lowercase=True, remove_punct=True, remove_stopwords=False),
    "Preserve case": dict(lowercase=False, remove_punct=False, remove_stopwords=False)
}

preprocessing_rows = []
for setting_name, kwargs in preprocessing_settings.items():
    for model_name, model in models.items():
        tokenizer = create_tokenizer(model, **kwargs)
        result = vocab_coverage(train_texts, tokenizer, model)
        preprocessing_rows.append({
            "Setting": setting_name,
            "Model": model_name,
            "Token coverage (%)": 100 * result["token_coverage"],
            "Unique coverage (%)": 100 * result["unique_coverage"]
        })

preprocessing_df = pd.DataFrame(preprocessing_rows)
display(preprocessing_df.round(2))

,Setting,Model,Token coverage (%),Unique coverage (%)
0,Baseline,WikiGiga,96.63,82.11
1,Baseline,Twitter,95.79,81.65
2,Remove punctuation,WikiGiga,97.56,87.91
3,Remove punctuation,Twitter,97.02,87.96
4,Preserve case,WikiGiga,86.64,61.87
5,Preserve case,Twitter,85.72,61.19


**Interpretation**

Removing punctuation increased token coverage from 96.63% to 97.56% for WikiGiga and from 95.79% to 97.02% for Twitter. Unique-token coverage also increased substantially for both models. This suggests that punctuation tokens and punctuation-containing expressions contribute to the OOV vocabulary.

However, higher coverage doesn't necessarily mean better emotion classification performance. Punctuation such as exclamation marks and repeated punctuation can convey emotional intensity. Removing it may therefore discard useful information.

Preserving capitalization substantially reduced token coverage to 86.64% for WikiGiga and 85.72% for Twitter. This indicates that lowercasing is important for matching tokens to these pretrained embedding vocabularies.

We therefore retain lowercasing while preserving punctuation as our baseline preprocessing. Although removing punctuation improves coverage, preserving it may retain useful emotional information. 

**1.4 Toke coverage per class**

Since our dataset is imbalanced and has 4 classes of emotions, we calculate the coverage separately for each class label: anger, joy, optimism and sadness.

In [87]:
# Compare WikiGiga and Twitter coverage separately for each emotion class.
class_coverage_rows = []
for model_name, model in models.items():
    tokenizer = create_tokenizer(model)
    for label_id, emotion in enumerate(label_names):
        emotion_texts = [
            text for text, label in zip(train_texts, labels)
            if label == label_id
        ]
        result = vocab_coverage(emotion_texts, tokenizer, model)
        class_coverage_rows.append({
            "Model": model_name,
            "Emotion": emotion,
            "Number of tweets": len(emotion_texts),
            "Token coverage (%)": 100 * result["token_coverage"],
            "Unique coverage (%)": 100 * result["unique_coverage"]
        })

class_coverage_df = pd.DataFrame(class_coverage_rows)
display(class_coverage_df.round(2))

,Model,Emotion,Number of tweets,Token coverage (%),Unique coverage (%)
0,WikiGiga,anger,1400,97.15,87.77
1,WikiGiga,joy,708,95.38,84.39
2,WikiGiga,optimism,294,96.76,89.66
3,WikiGiga,sadness,855,96.67,87.11
4,Twitter,anger,1400,96.21,86.45
5,Twitter,joy,708,94.61,84.11
6,Twitter,optimism,294,96.31,89.37
7,Twitter,sadness,855,95.85,86.35


Both embedding models achieve relatively high token coverage across all classes, ranging from 94.61% to 97.15%. WikiGiga consistently achieves slightly higher coverage than Twitter GloVe for every emotion class.

For both models, anger has the highest token coverage (97.15% for WikiGiga and 96.21% for Twitter), while joy has the lowest (95.38% and 94.61%). This suggests that tweets expressing joy contain a slightly larger proportion of tokens that are missing from the embedding vocabularies.

Overall, both models provide relatively consistent vocabulary coverage across emotion classes, although joy has somewhat lower coverage. WikiGiga consistently performs slightly better, suggesting that its vocabulary is at least as suitable for our dataset in terms of coverage alone.

# Question 2 
Task: Explore what your chosen word embedding model captures. Some things you might consider: 
- Is it good at representing interesting similarities (related to your choice of task)? 
- Have you found task-specific evidence that it's better than some other reasonable alternative?
- Do you know about limitations of this model?
- Any limitations that you find particularly troubling for your task? 

**Word-level emotion similarity**

We hypotethise that both embedding models will capture relationships between emotion related words, but GloVe Twitter may represent informal emotional language more effectively because it was pretrained on Twitter data. 

We first identify frequently occurring words in each emotion class in the TweetEval training set, excluding standard stopwords, punctuation, and dataset-specific tokens. From these frequency lists, we select representative emotion-related words that occur in our dataset and are present in both embedding vocabularies.


In [64]:
#identifying frequent words in the dataset
from collections import Counter
import string

tokenizer = create_tokenizer(
    wiki,
    lowercase=True,
    remove_punct=True,
    remove_stopwords=True
)

for label_id, emotion in enumerate(label_names):

    emotion_texts = [
        text for text, label in zip(train_texts, labels)
        if label == label_id
    ]

    counts = Counter()

    for text in emotion_texts:
        counts.update(tokenizer.tokens(text))

    print(f"\n{emotion.upper()}")

    for word, count in counts.most_common(20):
        print(f"{word}: {count}")


ANGER
user: 1010
like: 118
people: 101
get: 77
n: 68
angry: 65
fucking: 57
u: 56
amp: 55
anger: 52
got: 47
rage: 47
fuming: 46
one: 45
awful: 45
someone: 44
fuck: 43
even: 41
horrible: 40
would: 39

JOY
user: 459
n: 52
amp: 52
love: 51
amazing: 38
watch: 38
day: 37
hilarious: 37
like: 37
happy: 34
lol: 31
live: 29
one: 28
blues: 27
ly: 27
broadcast: 27
musically: 27
time: 25
get: 24
got: 22

OPTIMISM
user: 121
fear: 33
worry: 23
n: 22
get: 21
optimism: 20
good: 19
amp: 18
life: 16
never: 15
think: 14
like: 14
go: 13
start: 13
let: 13
would: 12
every: 11
quote: 11
want: 11
love: 10

SADNESS
user: 429
sad: 90
depression: 66
like: 59
sadness: 54
get: 47
n: 43
amp: 41
lost: 40
depressing: 39
feel: 36
life: 35
one: 34
anxiety: 33
really: 32
think: 31
know: 29
still: 28
day: 28
see: 27


Tokens such as 'user', 'n', and 'amp' occur frequently across all four emotion classes in the TweetEval dataset. Since these tokens provide little information for distinguishing emotion classes and may introduce noise into the representations, we exclude them from our analysis.

In particular, 'user' frequently comes from anonymized '@user' mentions, while 'amp' and 'n' may reflect tokenization artifacts. Removing these tokens reduces the influence of shared, non-emotional vocabulary when investigating emotion-related similarities.

In [68]:
excluded_tokens = {"user", "amp", "n"}

tokenizer = create_tokenizer(
    wiki,
    lowercase=True,
    remove_punct=True,
    remove_stopwords=True
)

emotion_counts = {}

for label_id, emotion in enumerate(label_names):

    counts = Counter()

    for text, label in zip(train_texts, labels):

        if label == label_id:
            tokens = tokenizer.tokens(text)

            counts.update(
                word for word in tokens
                if word not in excluded_tokens
            )

    emotion_counts[emotion] = counts

    print(f"\n{emotion.upper()}")
    print(counts.most_common(15))


ANGER
[('like', 118), ('people', 101), ('get', 77), ('angry', 65), ('fucking', 57), ('u', 56), ('anger', 52), ('got', 47), ('rage', 47), ('fuming', 46), ('one', 45), ('awful', 45), ('someone', 44), ('fuck', 43), ('even', 41)]

JOY
[('love', 51), ('amazing', 38), ('watch', 38), ('day', 37), ('hilarious', 37), ('like', 37), ('happy', 34), ('lol', 31), ('live', 29), ('one', 28), ('blues', 27), ('ly', 27), ('broadcast', 27), ('musically', 27), ('time', 25)]

OPTIMISM
[('fear', 33), ('worry', 23), ('get', 21), ('optimism', 20), ('good', 19), ('life', 16), ('never', 15), ('think', 14), ('like', 14), ('go', 13), ('start', 13), ('let', 13), ('would', 12), ('every', 11), ('quote', 11)]

SADNESS
[('sad', 90), ('depression', 66), ('like', 59), ('sadness', 54), ('get', 47), ('lost', 40), ('depressing', 39), ('feel', 36), ('life', 35), ('one', 34), ('anxiety', 33), ('really', 32), ('think', 31), ('know', 29), ('still', 28)]


In [69]:
#selecting emotion words:
emotion_words = {
    "anger": ["angry", "anger", "rage"],
    "joy": ["love", "amazing", "hilarious"],
    "optimism": ["optimism", "good", "fear"],
    "sadness": ["sad", "depression", "depressing"]
}

In [71]:
#checking the words are in both vocabularies
word_info = []

for emotion, words in emotion_words.items():
    for word in words:

        word_info.append({
            "Emotion": emotion,
            "Word": word,
            "Frequency": emotion_counts[emotion][word],
            "WikiGiga": word in wiki.vocab,
            "Twitter": word in twitter.vocab
        })

word_df = pd.DataFrame(word_info)

display(word_df)

,Emotion,Word,Frequency,WikiGiga,Twitter
0,anger,angry,65,True,True
1,anger,anger,52,True,True
2,anger,rage,47,True,True
3,joy,love,51,True,True
4,joy,amazing,38,True,True
5,joy,hilarious,37,True,True
6,optimism,optimism,20,True,True
7,optimism,good,19,True,True
8,optimism,fear,33,True,True
9,sadness,sad,90,True,True


**1. calculate cosine similarity between emotion words**

By calculating the cosine similarity, we investigate whether the model captures useful semantic relationships between the emotion related words.

- Within class similarity: Cosine similarity between tweets with the same emotion label.

- Between class similarity: Cosine similarity between tweets with different emotion labels.


In [72]:
word_pairs = [
    # Related emotional concepts
    ("angry", "anger", "related"),
    ("angry", "rage", "related"),
    ("anger", "rage", "related"),

    ("love", "amazing", "related"),
    ("amazing", "hilarious", "related"),

    ("sad", "depression", "related"),
    ("sad", "depressing", "related"),
    ("depression", "depressing", "related"),

    # Contrasting emotional concepts
    ("angry", "love", "contrasting"),
    ("anger", "good", "contrasting"),
    ("rage", "amazing", "contrasting"),

    ("sad", "love", "contrasting"),
    ("depressing", "good", "contrasting"),

    ("optimism", "fear", "contrasting"),
    ("fear", "good", "contrasting")
]

In [73]:
pair_results = []

for word1, word2, relationship in word_pairs:

    for name, emb_model in models.items():

        if word1 not in emb_model.vocab or word2 not in emb_model.vocab:
            print(f"Skipping {word1}, {word2} for {name}")
            continue

        v1 = emb_model.embed_word(word1)
        v2 = emb_model.embed_word(word2)

        similarity = cos_similarity(v1, v2)

        pair_results.append({
            "Word 1": word1,
            "Word 2": word2,
            "Relationship": relationship,
            "Model": name,
            "Cosine similarity": similarity
        })

pairs_df = pd.DataFrame(pair_results)

display(
    pairs_df.pivot(
        index=["Word 1", "Word 2", "Relationship"],
        columns="Model",
        values="Cosine similarity"
    ).round(4)
)

Model                               Twitter  WikiGiga
Word 1     Word 2     Relationship                   
amazing    hilarious  related        0.7451    0.6383
anger      good       contrasting    0.4986    0.5364
           rage       related        0.6708    0.8109
angry      anger      related        0.6205    0.8184
           love       contrasting    0.5888    0.5450
           rage       related        0.5907    0.6880
depressing good       contrasting    0.6210    0.4772
depression depressing related        0.5342    0.3964
fear       good       contrasting    0.5756    0.6705
love       amazing    related        0.8210    0.7521
optimism   fear       contrasting    0.6225    0.6399
rage       amazing    contrasting    0.4677    0.4451
sad        depressing related        0.7566    0.6491
           depression related        0.4945    0.3926
           love       contrasting    0.7393    0.6647

In [74]:
pair_summary = (
    pairs_df
    .groupby(["Model", "Relationship"])["Cosine similarity"]
    .mean()
    .unstack()
)

pair_summary["Difference"] = (
    pair_summary["related"] -
    pair_summary["contrasting"]
)

display(pair_summary.round(4))

Relationship,contrasting,related,Difference
Model,,,
Twitter,0.5877,0.6542,0.0665
WikiGiga,0.5684,0.6432,0.0748


**Interpreation**

Both models assign higher average cosine similarities to related word pairs than to contrasting pairs. For Twitter, the average similarity is 0.6542 for related pairs and 0.5877 for contrasting pairs, while WikiGiga achieves 0.6432 and 0.5684, respectively. This suggests that both models capture some meaningful semantic relationships between emotional words.

Twitter has slightly higher average similarities for both groups, whereas WikiGiga shows a slightly larger separation between related and contrasting pairs (0.0748 compared to 0.0665). However, this difference is small and does not provide strong evidence that one model is better at distinguishing emotions.

Overall, both embedding models represent some emotion-related similarities, but neither consistently separates related from contrasting emotional concepts. Since these comparisons involve a small, manually selected set of word pairs, they are exploratory and insufficient to determine which embedding model is better for our emotion classification task.

**2. Nearest neighbor experiment**

To further investigate what the pretrained embeddings capture we performed a nearest neighbour analysis using emotion-related words from our dataset.

Words with similar embeddings are expected to appear in similar contexts. By examining their nearest neighbours we can investigate whether the models capture meaningful emotion related similarities.

In [76]:
def nearest_neighbors(word, emb_model, k=5):

    if word not in emb_model.vocab:
        return None

    vector = emb_model.embed_word(word)

    nearest_ids = topk_ids(
        vector,
        emb_model.vectors,
        k=k + 3
    )

    neighbors = []

    for idx in nearest_ids:

        neighbor = emb_model.word(idx)

        if neighbor in {word, "<unk>", "<pad>"}:
            continue

        similarity = cos_similarity(
            vector,
            emb_model[idx]
        )

        neighbors.append((neighbor, similarity))

        if len(neighbors) == k:
            break

    return neighbors

In [77]:
neighbor_results = []

for emotion, words in emotion_words.items():

    for word in words:

        for model_name, emb_model in models.items():

            neighbors = nearest_neighbors(
                word,
                emb_model,
                k=5
            )

            if neighbors is None:
                print(f"{word} not in {model_name} vocabulary")
                continue

            for rank, (neighbor, similarity) in enumerate(
                neighbors, start=1
            ):

                neighbor_results.append({
                    "Emotion": emotion,
                    "Query": word,
                    "Model": model_name,
                    "Rank": rank,
                    "Neighbor": neighbor,
                    "Similarity": similarity
                })

neighbors_df = pd.DataFrame(neighbor_results)

display(neighbors_df.round(4))

,Emotion,Query,Model,Rank,Neighbor,Similarity
0,anger,angry,WikiGiga,1,enraged,0.8704
1,anger,angry,WikiGiga,2,furious,0.8629
2,anger,angry,WikiGiga,3,outraged,0.8448
3,anger,angry,WikiGiga,4,heard,0.8203
4,anger,angry,WikiGiga,5,anger,0.8184
...,...,...,...,...,...,...
115,sadness,depressing,Twitter,1,crappy,0.8572
116,sadness,depressing,Twitter,2,shitty,0.8459
117,sadness,depressing,Twitter,3,pointless,0.8378
118,sadness,depressing,Twitter,4,depressed,0.8370


In [78]:
neighbor_summary = (
    neighbors_df
    .groupby(["Emotion", "Query", "Model"])["Neighbor"]
    .apply(lambda words: ", ".join(words))
    .unstack("Model")
)

display(neighbor_summary)

Model                                                          Twitter  \
Emotion  Query                                                           
anger    anger        frustration, emotions, hatred, sadness, jealousy   
         angry                stupid, confused, birds, weird, annoying   
         rage                   force, blaze, violent, danger, torture   
joy      amazing      awesome, fantastic, great, incredible, wonderful   
         hilarious         funny, funniest, embarrassing, weird, jokes   
         love                            you, much, miss, loves, happy   
optimism fear                       afraid, lies, enemy, suffer, anger   
         good                           well, great, too, nice, better   
         optimism    uncertainty, urgency, enthusiasm, turmoil, opt...   
sadness  depressing    crappy, shitty, pointless, depressed, confusing   
         depression       anxiety, disorder, causes, sickness, disease   
         sad                       feel, feeling, bad, confused, upset   

Model                                                         WikiGiga  
Emotion  Query                                                          
anger    anger         frustration, outrage, resentment, fear, aroused  
         angry                enraged, furious, outraged, heard, anger  
         rage                  anger, fury, hatred, paranoia, hysteria  
joy      amazing     incredible, fantastic, wonderful, marvelous, a...  
         hilarious     amusing, comical, humorous, funny, melodramatic  
         love                         dreams, dream, loves, mind, life  
optimism fear                     worry, danger, fearing, fears, anger  
         good                  better, really, always, something, kind  
         optimism    pessimism, tempered, confidence, expectations,...  
sadness  depressing   unnerving, awful, unsettling, bleak, predictable  
         depression        onset, severe, experiencing, illness, fever  
         sad                 awful, sorry, feeling, horrible, terrible


**Interpretation**

The results show that both embedding models capture meaningful relationships between emotion-related words but the types of relationships differ.

WikiGiga often returned more direct emotional synonyms. For example, the nearest neighbours of angry included enraged, furious, and outraged, while rage was associated with anger, fury, and hatred. These results suggest that WikiGiga effectively represents closely related emotional concepts.

Twitter instead captured more informal or contextually related expressions. For example, the neighbours of amazing include awesome, fantastic, great, and incredible, while depressing is associated with informal negative expressions such as crappy, shitty, and pointless. These expressions are relevant to emotion classification because our dataset consists of informal tweets.

However, the results also reveal limitations. For example, Twitter associates angry with birds, which is not a direct emotional synonym. This may reflect contextual associations in the training data rather than emotional similarity. Additionally, WikiGiga associates optimism with pessimism, demonstrating that words with opposite meanings can be close in embedding space because they occur in similar contexts.

Overall, WikiGiga appears to capture more direct semantic relationships for several emotional words, while Twitter better represents some informal expressions commonly used in social media. However, neither model consistently represents emotional similarity and both sometimes produce contextually related words rather than synonyms. These findings suggest that Twitter GloVe may be useful for informal emotion classification.

**Limitations**

The results highlight an important limitation of distributional word embeddings: words that occur in similar contexts may receive similar representations even when they express contrasting emotions.

For example, WikiGiga returned pessimism among the nearest neighbours of optimism, while Twitter assigned a relatively high cosine similarity (0.6210) to depressing and good.

This is potentially problematic for emotion classification because the task requires distinguishing between contrasting emotional meanings.

Furthermore, GloVe produces static word embeddings, meaning that each word has a fixed representation regardless of context. This limits its ability to capture context-dependent emotional meanings.

Another limitaion is that 